Трет фајл од проектна задача -  истражување на GraphSAGE (Graph Sample and Aggregate)
===

Почитувани, во овој фајл го тестираме GraphSAGE. Во претходните фајлови експериментиравме со GCN и GAT. Една од работите што е многу важна но, за жал заборавив да ја споменам е тоа што овие три модели не се користат само за Link prediction туку и за node classification (да предвидиме во која категорија припаѓа некој јазол) или Graph classification (предвидуваме на која класа и припаѓа целиот граф).
    
    На кој начин точно функционира GraphSAGE?

Главната идеа кај GraphSAGE е дека не мора да ги обработуваме сите соседи на еден јазол. Можеме да земеме ограничен број на соседи и да ги искористиме нивните информации. Ова е особено корисно кај големи графови.



1.   Sample - зема примерок од соседите на некој јазол
2.   Aggregate - ги комбинира инфрмациите од избраните соседи
3.   Update - ги комбинира информациите од соседите со сопствените карактеристики на јазелот за да создаде нова репрезентација.





GraphSAGE е многу погоден за inductive learning.

    Што е inductive learning?

Inductive learning е начин на учење кај кој моделот може да научи општи правила за податоците за време на тренирањето а, потоа да ги примени и на нови податоци што претходно не ги видел.

Овде преку реален пример ќе покажеме како работи GraphSAGE:



> Имаме социјална мрежа, секој јазол претставува еден корисник а, врските меѓу јазлите претставуваат пријателства. Да претпоставиме и дека секој јазел (корисник) има 2 карактеристики: интерес за техлоногија и интерес за спорт.


download.svg

За секој јазол ја пишуваме неговата поврзаност со 2-те карактеристики. 1 ако е поврзан и 0 ако не е.

    Јазол	Технологија	Спорт

    A	        1	        0
    B	        1	        1
    C	        0	        1
    D	        1       	0
    E	        0	        0


Целен јазол ни е јазолот А. Почетната репрезентација на јазолот А = [1, 0].

    1) Сега првиот чекор е sampling односно не мора да ги земеме сите соседи на А. Ќе ги земеме само B и C.

    2) Вториот чекор е Aggregation. Сега треба да ги комбинираме карактеристиките на избраните соседи B и С.
    Во овој случај најлесно е да го користиме Mean Aggregator. Според податоците во табелата имаме: B = [1, 1] и С = [0, 1].
    Па од таму со Mean Agregator следува дека карактеристиките на соседите на А ќе имаат вектор [(1+0)/2, (1+1)/2].

    3) И во третиот чекор од GraphSAGЕ имаме Combine and Update односно не ги земаме во предвид само информациите од соседите туку ги комбинираме со оригиналните карактеристики на самиот јазол А. Правиме CONCAT на [0.5, 1] и [1, 0].

    Новата репрезентација на јазолот А ќе ги содржи информациите од јазолот А и неговите избрани соседи (В и С).



> Што се случува кога имаме повеќе слоеви во GraphSAGE?

Еве еден пример со 2 слоеви за да сватиме зошто бројот на слоеви е важен:



> Во првиот слој А добива информации од своите непосредни соседи.

> Во вториот слој добива и информации што тие соседи ги собрале од нивните соседи.

> Значи, со два слоја можеме да стигнеме до информации од јазли оддалечени два чекори од A.










In [1]:
!pip install torch-geometric --quiet

!pip install pyg-lib torch-scatter torch-sparse \
    -f https://data.pyg.org/whl/torch-2.11.0+cu130.html --quiet

!pip install ogb --upgrade --quiet
!pip install pyg_lib torch_scatter torch_sparse -f https://data.pyg.org/whl/torch-2.11.0+cu130.html

Looking in links: https://data.pyg.org/whl/torch-2.11.0+cu130.html


In [2]:
!pip show torch

Name: torch
Version: 2.11.0+cu130
Summary: Tensors and Dynamic neural networks in Python with strong GPU acceleration
Home-page: https://pytorch.org
Author: 
Author-email: PyTorch Team <packages@pytorch.org>
License: BSD-3-Clause
Location: /usr/local/lib/python3.13/dist-packages
Requires: cuda-bindings, cuda-toolkit, filelock, fsspec, jinja2, networkx, nvidia-cudnn-cu13, nvidia-cusparselt-cu13, nvidia-nccl-cu13, nvidia-nvshmem-cu13, setuptools, sympy, triton, typing-extensions
Required-by: accelerate, fastai, ogb, peft, sentence-transformers, timm, torchdata, torchvision


In [3]:
from ogb.linkproppred import PygLinkPropPredDataset
from torch_geometric.loader import DataLoader
import torch
from torch_geometric.data.data import DataEdgeAttr, DataTensorAttr
from torch_geometric.data.storage import GlobalStorage

/usr/local/lib/python3.13/dist-packages/outdated/__init__.py:36: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import parse_version


In [4]:
torch.serialization.add_safe_globals(
    [DataEdgeAttr, DataTensorAttr, GlobalStorage]
)

In [5]:
dataset = PygLinkPropPredDataset(name = "ogbl-ppa")

Downloaded 0.38 GB: 100%|██████████| 388/388 [01:18<00:00,  4.94it/s]


Extracting dataset/ppassoc.zip


Processing...


Loading necessary files...
This might take a while.
Processing graphs...


100%|██████████| 1/1 [00:00<00:00,  2.70it/s]


Converting graphs into PyG objects...


100%|██████████| 1/1 [00:00<00:00, 361.95it/s]


Saving...


Done!


In [6]:
print(dataset[0].x.shape)
print(dataset[0].edge_index.shape)

torch.Size([576289, 58])
torch.Size([2, 42463862])


In [7]:
import torch


if not hasattr(torch, "_real_load"):
    torch._real_load = torch.load

def trusted_load(*args, **kwargs):
    kwargs["weights_only"] = False

    return torch._real_load(*args, **kwargs)

torch.load = trusted_load




In [8]:
split_edge = dataset.get_edge_split()
graph = dataset[0]
graph.x = graph.x.float()
print(graph.x.dtype)

torch.float32


In [9]:
train_edge, valid_edge, test_edge = split_edge["train"], split_edge["valid"], split_edge["test"]

In [10]:
subset_size = 100000
train_edges_small = train_edge["edge"][:subset_size]
validation_edges_small = valid_edge["edge"][:20000]
validation_edges_neg_small = valid_edge["edge_neg"][:20000]
test_edges_small = test_edge["edge"][:20000]
test_edges_neg_small = test_edge["edge_neg"][:20000]

In [11]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
graph = graph.to(device)

In [20]:
from torch_geometric.nn import SAGEConv
import torch.nn.functional as F

class GraphSAGE(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, num_layers):
        super().__init__()

        self.convs = torch.nn.ModuleList()
        self.convs.append(
            SAGEConv(in_channels, hidden_channels))

        for _ in range(num_layers - 1):
            self.convs.append(
                SAGEConv(hidden_channels, hidden_channels)
            )

    def forward(self, x, edge_index):

        for i, conv in enumerate(self.convs):
            x = conv(x, edge_index)

            if i < len(self.convs) - 1:
                x = F.relu(x)

        return x

In [21]:
model_configs = {
    "GraphSAGE-2-layers-32": (2, 32),
    "GraphSAGE-2-layers-128": (2, 128),

    "GraphSAGE-3-layers-32": (3, 32),
    "GraphSAGE-3-layers-128": (3, 128),

    "GraphSAGE-4-layers-32": (4, 32),
    "GraphSAGE-4-layers-128": (4, 128)
}

In [22]:

from torch_geometric.loader import LinkNeighborLoader

train_loader = LinkNeighborLoader(
    data=graph,
    num_neighbors=[5, 5],
    edge_label_index=train_edges_small.t(),
    edge_label=torch.ones(train_edges_small.size(0)),
    batch_size=256,
    shuffle=True,
    neg_sampling_ratio=1.0
)

In [23]:
from torch_geometric.loader import LinkNeighborLoader

valid_loader = LinkNeighborLoader(
    data=graph,
    num_neighbors=[5, 5],

    edge_label_index=torch.cat([
        validation_edges_small.t(),
        validation_edges_neg_small.t()
    ], dim=1),

    edge_label=torch.cat([
        torch.ones(validation_edges_small.size(0)),
        torch.zeros(validation_edges_neg_small.size(0))
    ]),
    batch_size=256,
    shuffle=False

)

In [24]:
import time

def train(model, optimizer):
    model.train()

    start = time.time()
    total_loss = 0

    for batch in train_loader:
        batch = batch.to(device)

        optimizer.zero_grad()

        z = model(batch.x, batch.edge_index)

        edge_label_index = batch.edge_label_index
        edge_label = batch.edge_label.float()

        score = decode(z, edge_label_index.t())

        loss = torch.nn.functional.binary_cross_entropy_with_logits(
            score,
            edge_label
        )

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    sec = time.time() - start

    return avg_loss, sec

In [25]:
from sklearn.metrics import roc_auc_score

@torch.no_grad()
def evaluate(model):
    model.eval()

    all_scores = []
    all_labels = []

    for batch in valid_loader:
        batch = batch.to(device)

        z = model(batch.x, batch.edge_index)

        edge_label_index = batch.edge_label_index
        edge_label = batch.edge_label.float()

        score = decode(z, edge_label_index.t())
        prob = torch.sigmoid(score).cpu()

        all_scores.append(prob)
        all_labels.append(edge_label.cpu())

    y_score = torch.cat(all_scores)
    y_true = torch.cat(all_labels)

    auc = roc_auc_score(
        y_true.numpy(),
        y_score.numpy()
    )

    return auc

In [26]:
def decode(z, edge):
    src, dst = edge.t()
    return (z[src] * z[dst]).sum(dim=1)

In [27]:
import time

results = {}

learning_rates = [0.01, 0.001]
num_epochs = 50

for model_name, (num_layers, hidden_channels) in model_configs.items():

    for lr in learning_rates:

        experiment_name = f"{model_name}-lr-{lr}"

        print(f"\nTraining: {experiment_name}")

        # Create a new GraphSAGE model
        model = GraphSAGE(
            in_channels=58,
            hidden_channels=hidden_channels,
            num_layers=num_layers
        ).to(device)

        # Create a new optimizer
        optimizer = torch.optim.Adam(
            model.parameters(),
            lr=lr
        )

        best_auc = 0
        history = []

        for epoch in range(1, num_epochs + 1):

            epoch_start = time.time()

            # Training
            loss, train_time_sec = train(model, optimizer)

            # Validation
            val_auc = evaluate(model)

            epoch_total_sec = time.time() - epoch_start

            # Store epoch information
            history.append({
                "epoch": epoch,
                "train_loss": loss,
                "val_auc": val_auc,
                "train_time_sec": train_time_sec,
                "epoch_total_sec": epoch_total_sec
            })

            # Save the best model
            if val_auc > best_auc:
                best_auc = val_auc

                torch.save(
                    model.state_dict(),
                    f"best_{experiment_name}.pt"
                )

            print(
                f"Epoch {epoch:02d} | "
                f"Train Loss: {loss:.4f} | "
                f"Val AUC: {val_auc:.4f} | "
                f"Train Time: {train_time_sec:.2f}s | "
                f"Epoch Total: {epoch_total_sec:.2f}s"
            )

        # Store results of the current experiment
        results[experiment_name] = {
            "best_val_auc": best_auc,
            "history": history
        }

        print(f"Best Validation AUC: {best_auc:.4f}")


Training: GraphSAGE-2-layers-32-lr-0.01
Epoch 01 | Train Loss: 0.4952 | Val AUC: 0.8939 | Train Time: 9.34s | Epoch Total: 10.53s
Epoch 02 | Train Loss: 0.4805 | Val AUC: 0.8987 | Train Time: 5.71s | Epoch Total: 6.84s
Epoch 03 | Train Loss: 0.4766 | Val AUC: 0.9080 | Train Time: 6.31s | Epoch Total: 7.78s
Epoch 04 | Train Loss: 0.4733 | Val AUC: 0.9099 | Train Time: 9.45s | Epoch Total: 11.82s
Epoch 05 | Train Loss: 0.4688 | Val AUC: 0.9130 | Train Time: 6.72s | Epoch Total: 7.86s
Epoch 06 | Train Loss: 0.4697 | Val AUC: 0.9149 | Train Time: 6.35s | Epoch Total: 7.48s
Epoch 07 | Train Loss: 0.4685 | Val AUC: 0.9176 | Train Time: 5.68s | Epoch Total: 7.03s
Epoch 08 | Train Loss: 0.4656 | Val AUC: 0.9173 | Train Time: 6.14s | Epoch Total: 7.28s
Epoch 09 | Train Loss: 0.4639 | Val AUC: 0.9242 | Train Time: 6.06s | Epoch Total: 7.51s
Epoch 10 | Train Loss: 0.4629 | Val AUC: 0.9229 | Train Time: 5.77s | Epoch Total: 6.89s
Epoch 11 | Train Loss: 0.4627 | Val AUC: 0.9188 | Train Time: 6.29s

Во претходниот код направивме вкупно 12 експерименти на 6 GraphSAGE модели. Моделите беа:



1.   Модел со 2 скриени слоеви и 32 неврони во скриениот слој
2.   Модел со 2 скриени слоеви и 128 неврони во скриениот слој
3.   Модел со 3 скриени слоеви и 32 неврони во скриениот слој
4.   Модел со 3 скриени слоеви и 128 неврони во скриениот слој
5.   Модел со 4 скриени слоеви и 32 неврони во скриениот слој
6.   Модел со 4 скриени слоеви и 128 неврони во скриениот слој


Секој од овие модели при backpropagation ја пресметува грешката и користи optimizer за да направи промени во тежините и моделот да биде поточен. Овој optimizer е конфигуриран на 2 различни начини: со learning rate = [0.01, 0.001]. Затоа на сите 6 модели ние експериментираме и ги користиме и двете learning rate за да видиме која ќе даде подобри резултати.




In [28]:
# 1. Print best validation AUC for all 12 experiments
for name, result in results.items():
    print(f"{name}: {result['best_val_auc']:.4f}")


import pandas as pd
import zipfile
from google.colab import files

# 2. Collect training history from all experiments
history_rows = []

for experiment_name, result in results.items():
    for epoch_data in result["history"]:
        history_rows.append({
            "experiment": experiment_name,
            **epoch_data
        })

# 3. Save training history into a CSV file
graphsage_history = pd.DataFrame(history_rows)
graphsage_history.to_csv("graphsage_history.csv", index=False)

# 4. Put all best GraphSAGE models into one ZIP file
with zipfile.ZipFile("graphsage_best_models.zip", "w", zipfile.ZIP_DEFLATED) as zipf:
    for experiment_name in results:
        model_file = f"best_{experiment_name}.pt"
        zipf.write(model_file)

# 5. Download the files
files.download("graphsage_history.csv")
files.download("graphsage_best_models.zip")

GraphSAGE-2-layers-32-lr-0.01: 0.9271
GraphSAGE-2-layers-32-lr-0.001: 0.9313
GraphSAGE-2-layers-128-lr-0.01: 0.9269
GraphSAGE-2-layers-128-lr-0.001: 0.9419
GraphSAGE-3-layers-32-lr-0.01: 0.9223
GraphSAGE-3-layers-32-lr-0.001: 0.9347
GraphSAGE-3-layers-128-lr-0.01: 0.9214
GraphSAGE-3-layers-128-lr-0.001: 0.9484
GraphSAGE-4-layers-32-lr-0.01: 0.9226
GraphSAGE-4-layers-32-lr-0.001: 0.9365
GraphSAGE-4-layers-128-lr-0.01: 0.9220
GraphSAGE-4-layers-128-lr-0.001: 0.9488


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [29]:
from torch_geometric.loader import LinkNeighborLoader

test_loader = LinkNeighborLoader(
    data=graph,
    num_neighbors=[5, 5],

    edge_label_index=torch.cat([
        test_edges_small.t(),
        test_edges_neg_small.t()
    ], dim=1),

    edge_label=torch.cat([
        torch.ones(test_edges_small.size(0)),
        torch.zeros(test_edges_neg_small.size(0))
    ]),

    batch_size=256,
    shuffle=False
)

In [30]:
graphsage_models = {}

for model_name, (num_layers, hidden_channels) in model_configs.items():

    for lr in learning_rates:

        experiment_name = f"{model_name}-lr-{lr}"

        # Create the GraphSAGE model
        model = GraphSAGE(
            in_channels=58,
            hidden_channels=hidden_channels,
            num_layers=num_layers
        ).to(device)

        # Load the best saved parameters
        model.load_state_dict(
            torch.load(
                f"best_{experiment_name}.pt",
                map_location=device,
                weights_only=True
            )
        )

        # Set the model to evaluation mode
        model.eval()

        # Store the loaded model
        graphsage_models[experiment_name] = model

        print(f"Loaded: {experiment_name}")

Loaded: GraphSAGE-2-layers-32-lr-0.01
Loaded: GraphSAGE-2-layers-32-lr-0.001
Loaded: GraphSAGE-2-layers-128-lr-0.01
Loaded: GraphSAGE-2-layers-128-lr-0.001
Loaded: GraphSAGE-3-layers-32-lr-0.01
Loaded: GraphSAGE-3-layers-32-lr-0.001
Loaded: GraphSAGE-3-layers-128-lr-0.01
Loaded: GraphSAGE-3-layers-128-lr-0.001
Loaded: GraphSAGE-4-layers-32-lr-0.01
Loaded: GraphSAGE-4-layers-32-lr-0.001
Loaded: GraphSAGE-4-layers-128-lr-0.01
Loaded: GraphSAGE-4-layers-128-lr-0.001


In [31]:
from sklearn.metrics import roc_auc_score
import torch

@torch.no_grad()
def test(model):
    model.eval()

    all_scores = []
    all_labels = []

    for batch in test_loader:
        batch = batch.to(device)

        z = model(batch.x, batch.edge_index)

        edge_label_index = batch.edge_label_index
        edge_label = batch.edge_label.float()

        score = decode(z, edge_label_index.t())

        prob = torch.sigmoid(score)

        all_scores.append(prob.cpu())
        all_labels.append(edge_label.cpu())

    all_scores = torch.cat(all_scores)
    all_labels = torch.cat(all_labels)

    auc = roc_auc_score(
        all_labels.numpy(),
        all_scores.numpy()
    )

    return auc

In [32]:
test_results = {}

for experiment_name, model in graphsage_models.items():

    test_auc = test(model)

    test_results[experiment_name] = test_auc

    print(f"{experiment_name}: Test AUC = {test_auc:.4f}")

GraphSAGE-2-layers-32-lr-0.01: Test AUC = 0.9264
GraphSAGE-2-layers-32-lr-0.001: Test AUC = 0.9296
GraphSAGE-2-layers-128-lr-0.01: Test AUC = 0.9230
GraphSAGE-2-layers-128-lr-0.001: Test AUC = 0.9408
GraphSAGE-3-layers-32-lr-0.01: Test AUC = 0.9187
GraphSAGE-3-layers-32-lr-0.001: Test AUC = 0.9342
GraphSAGE-3-layers-128-lr-0.01: Test AUC = 0.9174
GraphSAGE-3-layers-128-lr-0.001: Test AUC = 0.9475
GraphSAGE-4-layers-32-lr-0.01: Test AUC = 0.9206
GraphSAGE-4-layers-32-lr-0.001: Test AUC = 0.9357
GraphSAGE-4-layers-128-lr-0.01: Test AUC = 0.9195
GraphSAGE-4-layers-128-lr-0.001: Test AUC = 0.9482


In [33]:
import pandas as pd
from google.colab import files

final_rows = []

for experiment_name, result in results.items():

    final_rows.append({
        "model": experiment_name,
        "best_val_auc": result["best_val_auc"],
        "test_auc": test_results[experiment_name],
        "epochs": 50
    })

final_metrics = pd.DataFrame(final_rows)

final_metrics.to_csv("graphsage_final_metrics.csv", index=False)

files.download("graphsage_final_metrics.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>